Viewed custom-layer.ipynb:1-403

本节 [custom-layer.ipynb](file:///Users/songchao/WorkSpace/PythonProject/DeepLearn/pytorch/chapter_deep-learning-computation/custom-layer.ipynb) 主要介绍了**如何在 PyTorch 中通过继承 `nn.Module` 构建自定义层（带参数与不带参数）**，以便实现深度学习框架中尚未内置的特殊操作。

---

### 一、核心内容总结

#### 1. 不带参数的自定义层
当自定义层不需要在训练中更新权重（例如特定的归一化、去中心化操作）时，只需实现 `forward` 函数：
* **示例（`CenteredLayer`）**：
  ```python
  class CenteredLayer(nn.Module):
      def __init__(self):
          super().__init__()

      def forward(self, X):
          return X - X.mean()
  ```
* **用法**：既可以直接接收张量调用（如 `layer(X)`），也可以无缝作为组件放入 `nn.Sequential` 中与其他层级联组合。

---

#### 2. 带参数的自定义层（使用 `nn.Parameter`）
当层包含可学习的参数（如权重 `weight` 和偏置 `bias`）时，需借助 `nn.Parameter` 封装张量：
* **示例（自定义带 ReLU 激活的全连接层 `MyLinear`）**：
  ```python
  class MyLinear(nn.Module):
      def __init__(self, in_units, units):
          super().__init__()
          # 使用 nn.Parameter 显式声明为模型参数
          self.weight = nn.Parameter(torch.randn(in_units, units))
          self.bias = nn.Parameter(torch.randn(units,))

      def forward(self, X):
          linear = torch.matmul(X, self.weight) + self.bias
          return F.relu(linear)
  ```
* **`nn.Parameter` 的核心作用**：
  * 自动将张量注册到当前模块的参数列表（`parameters()` / `named_parameters()`）中；
  * 自动开启梯度追踪（`requires_grad=True`）；
  * 便于优化器统一更新，以及模型的保存（`state_dict()`）与加载。

---

#### 3. 复合模型集成
无论是带参数还是不带参数的自定义层，其接口规范与 PyTorch 内置层（如 `nn.Linear`、`nn.Conv2d`）完全一致，可直接混合拼装：
```python
net = nn.Sequential(
    MyLinear(64, 8),
    CenteredLayer(),
    MyLinear(8, 1)
)
out = net(torch.rand(2, 64))
```

---

### 二、关键注意事项与工程实践

1. **避免在 `forward` 中直接使用 `.data`**：
   * 教程原代码中使用了 `self.weight.data`，但在实际训练中应直接使用 `self.weight`。访问 `.data` 会脱离计算图，导致 PyTorch 无法正常计算反向传播梯度。
2. **权重初始化**：
   * 实际构建自定义层时，可结合 `torch.nn.init`（如 Xavier、Kaiming 初始化）对 `self.weight` 进行更稳定的权重初始化。

---

### 三、课后练习要点

1. **张量降维双线性层**：
   * 计算公式：`y_k = ∑_{i, j} W_{ijk} · x_i · x_j`
   * 实现思路：定义形状为 `(in_features, in_features, out_features)` 的三维权重参数 `W`，在 `forward` 中使用 `torch.einsum('bi,bj,ijk->bk', x, x, self.weight)` 或双重矩阵乘法高效计算。
2. **傅里叶特征截断层**：
   * 实现思路：利用 `torch.fft.rfft` 对输入做实数快速傅里叶变换，截取前一半低频系数并输出其实部和虚部。

In [1]:
import torch
import torch.nn.functional as F
from torch import nn

In [2]:
class CenteredLayer(nn.Module):

    def __init__(self):
        super().__init__()
    
    def forward(self, X):
        return X - X.mean()

In [4]:
layter = CenteredLayer()
layter(torch.FloatTensor(range(5)))

tensor([-2., -1.,  0.,  1.,  2.])

In [5]:
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())
Y = net(torch.rand((4,8)))

In [6]:
Y.mean()

tensor(-4.1910e-09, grad_fn=<MeanBackward0>)

In [11]:
class MyLinear(nn.Module):

    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(size=(in_units, units)))
        self.bias = nn.Parameter(torch.randn(size=(units,)))

    def forward(self, X):
        linear = torch.matmul(X, self.weight) + self.bias
        return F.relu(linear)

In [12]:
linear = MyLinear(5,3)
linear.weight

Parameter containing:
tensor([[ 1.6531, -0.6009,  0.2970],
        [-0.9579, -0.1544, -0.7461],
        [ 1.4283,  0.6676,  0.1820],
        [-0.4800,  0.9265, -0.3907],
        [ 1.0291,  1.1196, -0.4160]], requires_grad=True)

In [13]:
linear(torch.rand(3,5))

tensor([[1.0545, 1.0298, 0.0000],
        [1.9293, 0.0000, 0.0000],
        [0.5019, 0.3148, 0.0000]], grad_fn=<ReluBackward0>)

In [10]:
net = nn.Sequential(MyLinear(8,64), MyLinear(64, 1))
net(torch.rand(2,8))

tensor([[4.2061],
        [5.3442]])